# Process a monocular *elevation* session

Same as `process-monocular-session.ipynb`, but for a session recorded with
`teensyStimulusMonocular`'s `opts.elevations` set (repeats the azimuth sweep at
several fixed screen heights instead of just the center). See
`GogglesDevice.has_elevation` / `MonocularDevice._get_repeats(eye=, elevation=)`.

Builds the `MonocularDevice`, then:
1. Runs the "regular" analysis restricted to the 120px (center) elevation --
   saved as `{session_name}_dataframe.pkl`, the same file/schema
   `monocular-disparity-distance-comparison.ipynb` already reads via
   `CrossExperimentAnalysis.from_devices(R, M, D)`.
2. Runs `SessionAnalyzer.analyze_by_elevation` to get every swept elevation,
   saved separately as `{session_name}_by_elevation_dataframe.pkl`, and plots
   response strength vs. elevation per eye.

Run this in its own kernel with **only** `disparity-analysis` on `sys.path`.

In [ ]:
import sys, os
import pandas as pd

base_dir = r"C:\Users\acostamendoza\projects\distance-project"
for p in (base_dir, base_dir + r"\disparity-analysis"):
    if p not in sys.path:
        sys.path.append(p)

from src.data.MonocularDevice import MonocularDevice
from src.analysis.CellAnalysis import SessionAnalyzer

%load_ext autoreload
%autoreload 2

In [ ]:
mouse, date, experiment = "SAM045", "260721", "goggles-monocular"   # <-- fill in your elevation session
M = MonocularDevice(mouse_name=mouse, record_date=date, experiment_name=experiment)
print("session:", M.session_name, "| eyes:", M.present_eyes, "| cells:", len(M.suite2p_id))
print("has_elevation:", M.has_elevation,
      "| elevations (px):", M.elevations.tolist() if M.has_elevation else None)

## 1. Regular analysis (120px / center elevation)

Same pkl filename/schema as the non-elevation notebook, so downstream
cross-experiment code keeps reading it unchanged. Falls back to no elevation
filter automatically if this session turns out not to have a sweep.

In [ ]:
elev_kwargs = {'elevation': 120.0} if M.has_elevation else {}
if M.has_elevation:
    print(f"Session swept elevations {M.elevations.tolist()} px -- using 120px as the 'regular' condition.")

pkl_path = os.path.join(M.data_dir, f"{M.session_name}_dataframe.pkl")
if os.path.exists(pkl_path):
    stats_df = pd.read_pickle(pkl_path)
    print(f"Loaded cached stats: {pkl_path}")
else:
    stats_df = SessionAnalyzer(M).analyze_session_stats(n_iterations=200, **elev_kwargs)
    SessionAnalyzer(M).save_dataframe(stats_df)   # -> pkl_path
stats_df.head()

## 2. Explore how responses change across elevation

One block of per-cell rows per swept elevation, concatenated into a single
long-format DataFrame (tagged by an `elevation` column). Only meaningful if
`M.has_elevation` is True.

In [ ]:
elev_pkl_path = os.path.join(M.data_dir, f"{M.session_name}_by_elevation_dataframe.pkl")
if os.path.exists(elev_pkl_path):
    by_elev_df = pd.read_pickle(elev_pkl_path)
    print(f"Loaded cached per-elevation stats: {elev_pkl_path}")
else:
    by_elev_df = SessionAnalyzer(M).analyze_by_elevation(n_iterations=200)
    by_elev_df.to_pickle(elev_pkl_path)
    print(f"Saved per-elevation stats: {elev_pkl_path}")
by_elev_df.head()

In [ ]:
import matplotlib.pyplot as plt

EYE_COLOR = {'left': 'red', 'right': 'blue'}   # matches CellVisualizer.EYE_COLOR

# NOTE: this filters each elevation block by that block's OWN reliability, which drops
# a cell's off-preferred elevations rather than showing its full tuning shape (including
# the weak-but-real responses away from peak). If you want the full curve per reliably-
# driven cell instead, filter once by the 120px-condition's `reliable` column in stats_df
# and then plot that same cell subset's rf_value across all elevations in by_elev_df.
for eye in M.present_eyes:
    reliable = by_elev_df[by_elev_df[f'{eye}_reliable']]
    summary = reliable.groupby('elevation')[f'{eye}_rf_value'].agg(['mean', 'sem'])
    plt.errorbar(summary.index, summary['mean'], yerr=summary['sem'],
                 marker='o', color=EYE_COLOR[eye], label=eye)

plt.xlabel('Elevation (px)')
plt.ylabel('Mean RF peak response (z-score)')
plt.title(f'{M.session_name} -- response vs elevation by eye')
plt.legend(frameon=False)
plt.show()